# MarketPulse – Market Analytics Notebook

This notebook explores daily market data for US equities and indices.
It covers data validation, analytics views, and signal generation.

## Table of Contents
1. Data Sanity Check  
2. Daily Prices View  
3. Daily Returns  
4. Technical Features (MA, Volatility)  
5. Market Movers  
6. Trend States  
7. Signals (MA Cross)

In [3]:
import duckdb

con = duckdb.connect("../data/db/marketpulse.duckdb")


con.execute("""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT ticker) AS tickers,
  MIN(date) AS min_date,
  MAX(date) AS max_date
FROM raw_prices
""").fetchdf()


,rows,tickers,min_date,max_date
0,2259,9,2025-01-06,2026-01-06


In [4]:
con.execute("""
CREATE OR REPLACE VIEW prices_daily AS
SELECT
  date,
  ticker,
  open,
  high,
  low,
  close,
  adj_close,
  volume
FROM raw_prices
""")


In [5]:
con.execute("""
CREATE OR REPLACE VIEW returns_daily AS
SELECT
  date,
  ticker,
  close,
  close / LAG(close) OVER (PARTITION BY ticker ORDER BY date) - 1 AS daily_return
FROM prices_daily
""")


In [6]:
con.execute("""
CREATE OR REPLACE VIEW features_daily AS
SELECT
  date,
  ticker,
  close,

  AVG(close) OVER (
    PARTITION BY ticker
    ORDER BY date
    ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
  ) AS ma_20,

  AVG(close) OVER (
    PARTITION BY ticker
    ORDER BY date
    ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
  ) AS ma_50,

  STDDEV_SAMP(daily_return) OVER (
    PARTITION BY ticker
    ORDER BY date
    ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
  ) AS vol_20

FROM returns_daily
""")


In [8]:
### Top movers – מי זז הכי חזק היום?

con.execute("""
WITH latest_day AS (
  SELECT MAX(date) AS d FROM returns_daily
)
SELECT
  r.date,
  r.ticker,
  ROUND(r.daily_return * 100, 2) AS daily_return_pct
FROM returns_daily r
JOIN latest_day l ON r.date = l.d
WHERE r.daily_return IS NOT NULL
ORDER BY ABS(r.daily_return) DESC
LIMIT 10
""").fetchdf()


,date,ticker,daily_return_pct
0,2026-01-06,AMZN,2.35
1,2026-01-06,AAPL,-1.67
2,2026-01-06,GOOGL,-1.18
3,2026-01-06,NVDA,0.54
4,2026-01-06,DIA,0.42
5,2026-01-06,QQQ,0.34
6,2026-01-06,MSFT,-0.30
7,2026-01-06,SPY,0.16
8,2026-01-06,IWM,-0.01


In [ ]:
### Trend state – MA20 מול MA50 (ביום האחרון)


con.execute("""
WITH latest_day AS (
  SELECT MAX(date) AS d FROM features_daily
)
SELECT
  f.date,
  f.ticker,
  ROUND(f.close, 2) AS close,
  ROUND(f.ma_20, 2) AS ma_20,
  ROUND(f.ma_50, 2) AS ma_50,
  CASE
    WHEN f.ma_20 > f.ma_50 THEN 'UPTREND'
    WHEN f.ma_20 < f.ma_50 THEN 'DOWNTREND'
    ELSE 'FLAT'
  END AS trend_state
FROM features_daily f
JOIN latest_day l ON f.date = l.d
ORDER BY f.ticker
""").fetchdf()


,date,ticker,close,ma_20,ma_50,trend_state
0,2026-01-06,AAPL,262.79,273.34,273.07,UPTREND
1,2026-01-06,AMZN,238.54,229.13,232.18,DOWNTREND
2,2026-01-06,DIA,491.84,483.68,476.97,UPTREND
3,2026-01-06,GOOGL,312.80,311.52,300.38,UPTREND
4,2026-01-06,IWM,252.70,251.27,246.40,UPTREND
5,2026-01-06,MSFT,471.41,482.19,493.67,DOWNTREND
6,2026-01-06,NVDA,189.14,183.54,186.81,DOWNTREND
7,2026-01-06,QQQ,620.07,618.00,616.85,UPTREND
8,2026-01-06,SPY,688.83,684.16,679.48,UPTREND


In [11]:
### Volatility leaders – מי הכי תנודתי (Vol20) ביום האחרון


con.execute("""
WITH latest_day AS (
  SELECT MAX(date) AS d FROM features_daily
)
SELECT
  f.date,
  f.ticker,
  ROUND(f.vol_20 * 100, 2) AS vol_20_pct
FROM features_daily f
JOIN latest_day l ON f.date = l.d
WHERE f.vol_20 IS NOT NULL
ORDER BY f.vol_20 DESC
LIMIT 10
""").fetchdf()


,date,ticker,vol_20_pct
0,2026-01-06,NVDA,1.88
1,2026-01-06,AMZN,1.40
2,2026-01-06,GOOGL,1.38
3,2026-01-06,MSFT,1.06
4,2026-01-06,IWM,0.92
5,2026-01-06,QQQ,0.85
6,2026-01-06,AAPL,0.69
7,2026-01-06,DIA,0.62
8,2026-01-06,SPY,0.56


In [12]:
con.execute("""
CREATE OR REPLACE VIEW signals_daily AS
WITH base AS (
  SELECT
    date,
    ticker,
    close,
    ma_20,
    ma_50,
    (ma_20 - ma_50) AS spread
  FROM features_daily
)
SELECT
  date,
  ticker,
  close,
  ma_20,
  ma_50,
  spread,
  LAG(spread) OVER (PARTITION BY ticker ORDER BY date) AS prev_spread,
  CASE
    WHEN LAG(spread) OVER (PARTITION BY ticker ORDER BY date) IS NULL THEN NULL
    WHEN LAG(spread) OVER (PARTITION BY ticker ORDER BY date) <= 0 AND spread > 0 THEN 'BULL_CROSS'
    WHEN LAG(spread) OVER (PARTITION BY ticker ORDER BY date) >= 0 AND spread < 0 THEN 'BEAR_CROSS'
    ELSE NULL
  END AS signal
FROM base
""")


In [13]:
con.execute("""
SELECT *
FROM signals_daily
WHERE signal IS NOT NULL
ORDER BY date DESC
LIMIT 20
""").fetchdf()


,date,ticker,close,ma_20,ma_50,spread,prev_spread,signal
0,2025-12-17,QQQ,600.409973,613.656497,613.560798,0.095699,-0.191301,BULL_CROSS
1,2025-12-15,IWM,251.929993,245.455499,245.036200,0.419300,-0.181200,BULL_CROSS
2,2025-12-12,AMZN,226.190002,228.567001,229.160200,-0.593200,0.051900,BEAR_CROSS
3,2025-12-09,QQQ,625.049988,612.175000,612.265599,-0.090599,0.344800,BEAR_CROSS
4,2025-12-02,NVDA,181.460007,186.309002,186.851801,-0.542799,0.685201,BEAR_CROSS
5,2025-11-19,MSFT,487.119995,513.150504,514.087599,-0.937095,0.468904,BEAR_CROSS
6,2025-11-18,IWM,233.470001,243.102999,243.104400,-0.001401,0.607000,BEAR_CROSS
7,2025-11-06,AMZN,243.039993,227.391500,227.130000,0.261500,-0.225099,BULL_CROSS
8,2025-10-08,MSFT,524.849976,514.338997,513.738599,0.600398,-0.378001,BULL_CROSS
9,2025-10-06,AMZN,220.899994,225.907500,226.390399,-0.482900,0.053300,BEAR_CROSS
